In [0]:
from pyspark.sql import functions as F

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")
print("Gold schema ready.")

In [0]:

df_dim_store = spark.table("workspace.silver.store")

df_dim_store.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("workspace.gold.dim_store")

print(f"dim_store: {df_dim_store.count()} rows, {len(df_dim_store.columns)} columns")
display(df_dim_store.limit(5))

In [0]:

df_order = spark.table("workspace.silver.order")

df_dim_technician = (
    df_order
    .filter(F.col("technician_id").isNotNull())
    .select("technician_id", "technician_name")
    .dropDuplicates(["technician_id"])
    .orderBy("technician_id")
)

df_dim_technician.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("workspace.gold.dim_technician")

print(f"dim_technician: {df_dim_technician.count()} rows")
display(df_dim_technician.limit(5))

In [0]:

df_estimate = spark.table("workspace.silver.estimate")

df_dim_estimator = (
    df_estimate
    .filter(F.col("estimator_id").isNotNull())
    .select("estimator_id", "estimator_name")
    .dropDuplicates(["estimator_id"])
    .orderBy("estimator_id")
)

df_dim_estimator.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("workspace.gold.dim_estimator")

print(f"dim_estimator: {df_dim_estimator.count()} rows")
display(df_dim_estimator.limit(5))

In [0]:

df_fact_order = (
    spark.table("workspace.silver.order")
    .select(
        "order_id",
        "store_id",                     
        "technician_id",                
        "service_type",
        "vehicle_no", "vehicle_make", "vehicle_model",
        "vehicle_in_datetime", "vehicle_out_datetime",
        "planned_work_start_datetime", "actual_work_start_datetime",
        "planned_completion_datetime", "actual_completion_datetime",
        "promised_delivery_datetime", "actual_delivery_datetime",
        "order_status",
        "customer_name", "customer_phone"
    )
)


df_fact_order.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("workspace.gold.fact_order")

print(f"fact_order: {df_fact_order.count()} rows, {len(df_fact_order.columns)} columns")
display(df_fact_order.limit(5))

In [0]:

df_fact_survey = spark.table("workspace.silver.customer_survey")

df_fact_survey.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("workspace.gold.fact_survey")

print(f"fact_survey: {df_fact_survey.count()} rows, {len(df_fact_survey.columns)} columns")
display(df_fact_survey.limit(5))

In [0]:

df_fact_estimate = (
    spark.table("workspace.silver.estimate")
    .select(
        "estimate_id",
        "order_id",                        
        "version_no",
        "estimate_amount",
        "created_at",
        "created_by",
        "estimate_type",
        "estimator_id"                     
    )
)


df_fact_estimate.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("workspace.gold.fact_estimate")

print(f"fact_estimate: {df_fact_estimate.count()} rows, {len(df_fact_estimate.columns)} columns")
display(df_fact_estimate.limit(5))

In [0]:

df_fact_invoice = spark.table("workspace.silver.invoice")

df_fact_invoice.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("workspace.gold.fact_invoice")

print(f"fact_invoice: {df_fact_invoice.count()} rows, {len(df_fact_invoice.columns)} columns")
display(df_fact_invoice.limit(5))

In [0]:

df_fact_budget = spark.table("workspace.silver.ns_budget")

df_fact_budget.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("workspace.gold.fact_budget")

print(f"fact_budget: {df_fact_budget.count()} rows, {len(df_fact_budget.columns)} columns")
display(df_fact_budget.limit(5))

In [0]:
import builtins


gold_tables = [
    ("dim_store", "DIMENSION"), ("dim_technician", "DIMENSION"), ("dim_estimator", "DIMENSION"),
    ("fact_order", "FACT"), ("fact_survey", "FACT"), ("fact_estimate", "FACT"),
    ("fact_invoice", "FACT"), ("fact_budget", "FACT")
]

results = []
for table, ttype in gold_tables:
    df = spark.table(f"workspace.gold.{table}")
    results.append((ttype, table, df.count(), len(df.columns), ", ".join(df.columns)))

import pandas as pd
pdf = pd.DataFrame(results, columns=["type", "table", "rows", "columns", "column_names"])
display(pdf)